# CNN 레이어 모듈

이 노트북은 `src/nn/conv.py`가 제공하는 CNN 레이어 모듈과 `im2col`/`col2im` 변환을 직접 실행하고 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요; CuPy 없으면 numpy fallback)

**목표**
- `im2col`의 shape 변환 원리와 출력 크기 수식을 확인한다.
- `Conv2d`, `MaxPool2d`, `Flatten`, `Dropout` 레이어의 forward/backward shape를 추적한다.
- CNN 전체 아키텍처(Conv→Pool→Flatten→Linear)를 조립하고 단계별 shape를 확인한다.
- CuPy 환경 여부에 따른 numpy/cupy fallback 패턴을 확인한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# CuPy가 없으면 numpy로 fallback
try:
    import cupy as cp
    xp = cp
    print(f"CuPy available: {cp.__version__}")
except ImportError:
    xp = np
    print("CuPy not available → using numpy (CPU)")

In [ ]:
from src.nn.conv import im2col, col2im, Conv2d, MaxPool2d, Flatten, Dropout
from src.nn.layers import Linear, ReLU, Sequential

## 1. 개요

`src/nn/conv.py`는 CNN을 구성하는 레이어 모듈과 핵심 변환 함수를 제공한다.
모든 레이어는 `Module` 기반이며, `xp` 파라미터로 numpy(CPU)와 cupy(GPU) 전환이 가능하다.

| 이름 | 역할 |
|---|---|
| `im2col` | `(B, C, H, W)` → `(B·out_h·out_w, C·K·K)` 행렬 변환 |
| `col2im` | `im2col` 역변환, Conv2d backward에서 `dx` 복원 |
| `Conv2d` | 2D 합성곱 레이어 |
| `MaxPool2d` | 2D max pooling 레이어 |
| `Flatten` | `(B, C, H, W)` → `(B, C·H·W)` |
| `Dropout` | inverted dropout |

**출력 크기 수식**:

$$
out\_h = \left\lfloor \frac{H + 2 \times padding - K}{stride} \right\rfloor + 1, \qquad
out\_w = \left\lfloor \frac{W + 2 \times padding - K}{stride} \right\rfloor + 1
$$

## 2. im2col 원리

직접 루프로 합성곱을 구현하면 4중 이상의 중첩 루프가 필요하다.
`im2col`은 커널이 슬라이딩하는 모든 위치의 입력 패치를 행으로 펼쳐 2D 행렬로 변환한다.
이 행렬에 커널 가중치 행렬을 곱하면 합성곱 전체를 단일 행렬 곱으로 계산할 수 있다.

| 단계 | shape | 설명 |
|---|---|---|
| 입력 `x` | `(B, C, H, W)` | 배치 이미지 |
| `im2col` 출력 | `(B·out_h·out_w, C·K·K)` | 각 행이 하나의 커널 위치 패치 |
| 커널 reshape | `(out_c, C·K·K)` | 각 행이 하나의 출력 채널 필터 |
| 행렬 곱 결과 | `(B·out_h·out_w, out_c)` | 각 행이 하나의 출력 공간 위치 |
| reshape 최종 | `(B, out_c, out_h, out_w)` | feature map |

In [ ]:
# im2col shape 확인
B, C, H, W = 2, 1, 6, 6
K = 3

x = np.random.randn(B, C, H, W).astype(np.float32)
col, out_h, out_w = im2col(x, kernel_size=K, stride=1, padding=0)

expected_out_h = (H - K) // 1 + 1  # = 4
expected_out_w = (W - K) // 1 + 1  # = 4

print(f"input shape:  {x.shape}")
print(f"col shape:    {col.shape}  <- (B*out_h*out_w, C*K*K) = ({B}*{expected_out_h}*{expected_out_w}, {C}*{K}*{K})")
print(f"out_h={out_h}, out_w={out_w}  (expected: {expected_out_h}, {expected_out_w})")

In [ ]:
# padding=1로 입력 크기 유지
col_pad, out_h_p, out_w_p = im2col(x, kernel_size=3, stride=1, padding=1)
print(f"padding=1: col shape = {col_pad.shape}, out_h={out_h_p}, out_w={out_w_p}")
print(f"  → 입력과 동일한 공간 크기 유지: {out_h_p == H} (H={H})")

# stride=2
col_s2, out_h_s2, out_w_s2 = im2col(x, kernel_size=3, stride=2, padding=0)
print(f"stride=2:  col shape = {col_s2.shape}, out_h={out_h_s2}, out_w={out_w_s2}")

In [ ]:
# im2col 값 검증: 첫 번째 행은 첫 번째 패치 위치(0,0)의 픽셀
x_simple = np.arange(1*1*4*4, dtype=np.float32).reshape(1, 1, 4, 4)
col_s, _, _ = im2col(x_simple, kernel_size=2, stride=1, padding=0)

print("input (4x4):")
print(x_simple[0, 0])
print(f"\nim2col shape: {col_s.shape}  <- 9 patches, each 2x2=4 elements")
print("첫 번째 패치 (위치 0,0):")
print(f"  col[0] = {col_s[0]}  <- [0,1,4,5] 기대")

## 3. col2im

`col2im`은 `im2col`의 역연산으로, Conv2d backward에서 입력에 대한 gradient `dx`를 복원한다.
동일한 입력 픽셀이 여러 패치에 반복 등장하므로 `+=`로 각 위치의 gradient를 누적 합산한다.

In [ ]:
# col2im shape 검증
B, C, H, W = 2, 3, 8, 8
K = 3
x_orig = np.random.randn(B, C, H, W).astype(np.float32)
col_orig, out_h, out_w = im2col(x_orig, kernel_size=K, stride=1, padding=1)

dx = col2im(col_orig, x_orig.shape, kernel_size=K, stride=1, padding=1)
print(f"원본 x shape:  {x_orig.shape}")
print(f"col shape:     {col_orig.shape}")
print(f"dx shape:      {dx.shape}  <- 원본과 동일해야 함")
print(f"shape 일치:    {dx.shape == x_orig.shape}")

## 4. Conv2d

2D 합성곱 레이어이다. `im2col`로 입력을 행렬로 펼친 뒤 커널 가중치와 행렬 곱을 수행한다.

**He 초기화**: $W \sim \mathcal{N}(0,\ \sqrt{2/(C \cdot K \cdot K)})$

In [ ]:
B = 4
x_img = np.random.randn(B, 1, 28, 28).astype(np.float32)  # MNIST 형태

conv = Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1, seed=42, xp=np)

print("=== Conv2d(1, 8, kernel_size=3, padding=1) ===")
print(f"w shape: {conv.w.shape}  <- (out_c, in_c, K, K)")
print(f"b shape: {conv.b.shape}  <- (out_c,)")

out_conv = conv.forward(x_img)
print(f"\ninput  shape: {x_img.shape}")
print(f"output shape: {out_conv.shape}  <- (B, 8, 28, 28) 기대 (padding=1로 크기 유지)")

In [ ]:
dout_conv = np.random.randn(*out_conv.shape).astype(np.float32)
dx_conv = conv.backward(dout_conv)

print(f"dout shape:   {dout_conv.shape}")
print(f"dx shape:     {dx_conv.shape}   <- 입력과 동일해야 함")
print(f"grad_w shape: {conv.grad_w.shape}")
print(f"grad_b shape: {conv.grad_b.shape}")
print(f"dx/input shape 일치: {dx_conv.shape == x_img.shape}")

## 5. MaxPool2d

2D max pooling 레이어이다. `im2col`로 각 풀링 윈도우를 행으로 펼친 뒤 행별 최대값을 선택한다.
forward에서 `_max_indices`를 저장하여 backward에서 최대값 위치에만 gradient를 전달한다.

$$
\text{out}[b, c, i, j] = \max_{k_h, k_w} x[b, c,\ i \cdot s + k_h,\ j \cdot s + k_w]
$$

In [ ]:
pool = MaxPool2d(kernel_size=2, xp=np)

# Conv2d 출력을 pooling
out_pool = pool.forward(out_conv)
print(f"input  shape: {out_conv.shape}")
print(f"output shape: {out_pool.shape}  <- (B, 8, 14, 14) 기대 (28/2=14)")

dout_pool = np.random.randn(*out_pool.shape).astype(np.float32)
dx_pool = pool.backward(dout_pool)
print(f"dx shape:     {dx_pool.shape}   <- 입력과 동일해야 함")
print(f"dx/input shape 일치: {dx_pool.shape == out_conv.shape}")

In [ ]:
# max 위치에만 gradient가 전달되는지 확인
x_tiny = np.array([[[[1.0, 4.0],
                     [3.0, 2.0]]]], dtype=np.float32)  # (1, 1, 2, 2)
pool_tiny = MaxPool2d(kernel_size=2, xp=np)
out_tiny = pool_tiny.forward(x_tiny)
dx_tiny = pool_tiny.backward(np.ones_like(out_tiny))

print(f"input:\n{x_tiny[0,0]}")
print(f"max output: {out_tiny[0,0,0,0]}  <- 4.0 기대")
print(f"dx:\n{dx_tiny[0,0]}")
print("→ 최대값 위치(0,1)에만 gradient=1, 나머지=0")

## 6. Flatten

4D 텐서를 2D로 펼친다. CNN feature map을 fully-connected 레이어에 연결하기 위해 사용한다.

$(B, C, H, W) \to (B, C \cdot H \cdot W)$

In [ ]:
flat = Flatten()
out_flat = flat.forward(out_pool)

print(f"input  shape: {out_pool.shape}")
print(f"output shape: {out_flat.shape}  <- (B, 8*14*14) = (B, 1568) 기대")

dout_flat = np.random.randn(*out_flat.shape).astype(np.float32)
dx_flat = flat.backward(dout_flat)
print(f"dx shape:     {dx_flat.shape}   <- 입력과 동일해야 함")
print(f"dx/input shape 일치: {dx_flat.shape == out_pool.shape}")

## 7. Dropout

학습 중 무작위로 뉴런을 비활성화하는 정규화 레이어이다. inverted dropout 방식을 사용한다.

$$
\text{mask}[i] = \begin{cases} 0 & \text{확률 } p \\ \dfrac{1}{1-p} & \text{확률 } 1-p \end{cases}, \qquad \text{out} = x \cdot \text{mask}
$$

평가 모드에서는 마스크를 적용하지 않고 입력을 그대로 통과시킨다.

In [ ]:
np.random.seed(0)
drop = Dropout(p=0.5)

x_d = np.ones((1000, 10), dtype=np.float32)

# 학습 모드
drop.train()
out_train = drop.forward(x_d)
active_ratio = (out_train != 0).mean()
print(f"training mode:")
print(f"  active ratio: {active_ratio:.3f}  (expected ~0.5)")
print(f"  active output mean: {out_train[out_train != 0].mean():.3f}  (expected ~2.0, inverted scaling)")

# 평가 모드
drop.eval()
out_eval = drop.forward(x_d)
print(f"\neval mode:")
print(f"  output == input: {np.array_equal(out_eval, x_d)}  <- 입력 그대로 반환")

## 8. CNN 아키텍처 shape 추적

MNIST 입력 `(B, 1, 28, 28)`에 대해 Conv→Pool→Flatten→Linear 전체 경로의 shape를 단계별로 확인한다.

In [ ]:
B = 4
x_mnist = np.random.randn(B, 1, 28, 28).astype(np.float32)

# 아키텍처 구성
layers = [
    ("Conv2d(1→8, K=3, pad=1)",  Conv2d(1,  8, kernel_size=3, padding=1, seed=0, xp=np)),
    ("ReLU",                       ReLU()),
    ("MaxPool2d(K=2)",             MaxPool2d(kernel_size=2, xp=np)),
    ("Conv2d(8→16, K=3, pad=1)",  Conv2d(8, 16, kernel_size=3, padding=1, seed=1, xp=np)),
    ("ReLU",                       ReLU()),
    ("MaxPool2d(K=2)",             MaxPool2d(kernel_size=2, xp=np)),
    ("Flatten",                    Flatten()),
    ("Linear(784→10)",             Linear(16 * 7 * 7, 10, seed=2)),
]

print(f"{'Layer':<30} {'Input shape':<22} {'Output shape':<22}")
print("-" * 75)
h = x_mnist
print(f"{'(input)':<30} {'-':<22} {str(h.shape):<22}")
for name, layer in layers:
    in_shape = h.shape
    h = layer.forward(h)
    print(f"{name:<30} {str(in_shape):<22} {str(h.shape):<22}")

print(f"\n최종 logits shape: {h.shape}  <- (B, 10) 기대")

In [ ]:
# shape 변화 시각화
stage_names = ["Input", "Conv1+ReLU", "Pool1", "Conv2+ReLU", "Pool2", "Flatten", "Linear"]
spatial_sizes = [28, 28, 14, 14, 7, None, None]
channel_counts = [1, 8, 8, 16, 16, 16*7*7, 10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# spatial size
valid_spatial = [(i, s) for i, s in enumerate(spatial_sizes) if s is not None]
idx_s, sizes_s = zip(*valid_spatial)
axes[0].bar(idx_s, sizes_s, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(len(stage_names)))
axes[0].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=9)
axes[0].set_title("Spatial Size (H or W)")
axes[0].set_ylabel("size")
axes[0].grid(axis='y', alpha=0.3)
for i, s in valid_spatial:
    axes[0].text(i, s + 0.3, str(s), ha='center', fontsize=9)

# channel / feature count
axes[1].bar(range(len(channel_counts)), channel_counts, color='tomato', alpha=0.8)
axes[1].set_xticks(range(len(stage_names)))
axes[1].set_xticklabels(stage_names, rotation=30, ha='right', fontsize=9)
axes[1].set_title("Channel / Feature Dimension")
axes[1].set_ylabel("count")
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)
for i, c in enumerate(channel_counts):
    axes[1].text(i, c * 1.1, str(c), ha='center', fontsize=9)

fig.suptitle("CNN Architecture — Shape Tracking", fontsize=13)
fig.tight_layout()
plt.show()

## 9. 검증

In [ ]:
B = 2
x_t = np.random.randn(B, 1, 28, 28).astype(np.float32)

# im2col
col_t, oh, ow = im2col(x_t, kernel_size=3, stride=1, padding=1)
assert col_t.shape == (B * oh * ow, 1 * 3 * 3), f"im2col shape mismatch: {col_t.shape}"
assert oh == 28 and ow == 28, f"expected out_h=out_w=28, got {oh},{ow}"

# col2im
dx_t = col2im(col_t, x_t.shape, kernel_size=3, stride=1, padding=1)
assert dx_t.shape == x_t.shape, f"col2im shape mismatch: {dx_t.shape}"

# Conv2d
conv_t = Conv2d(1, 4, kernel_size=3, padding=1, seed=0, xp=np)
out_c = conv_t.forward(x_t)
assert out_c.shape == (B, 4, 28, 28), f"Conv2d output shape: {out_c.shape}"
dx_c = conv_t.backward(np.ones_like(out_c))
assert dx_c.shape == x_t.shape, f"Conv2d dx shape: {dx_c.shape}"
assert conv_t.grad_w.shape == conv_t.w.shape

# MaxPool2d
pool_t = MaxPool2d(kernel_size=2, xp=np)
out_p = pool_t.forward(out_c)
assert out_p.shape == (B, 4, 14, 14), f"MaxPool2d output shape: {out_p.shape}"
dx_p = pool_t.backward(np.ones_like(out_p))
assert dx_p.shape == out_c.shape, f"MaxPool2d dx shape: {dx_p.shape}"

# Flatten
flat_t = Flatten()
out_f = flat_t.forward(out_p)
assert out_f.shape == (B, 4 * 14 * 14), f"Flatten output shape: {out_f.shape}"
dx_f = flat_t.backward(np.ones_like(out_f))
assert dx_f.shape == out_p.shape, f"Flatten dx shape: {dx_f.shape}"

# Dropout
drop_t = Dropout(p=0.5)
drop_t.train()
x_drop = np.ones((100, 10), dtype=np.float32)
out_d = drop_t.forward(x_drop)
assert out_d.shape == x_drop.shape
drop_t.eval()
out_d_eval = drop_t.forward(x_drop)
assert np.array_equal(out_d_eval, x_drop), "eval mode must pass input unchanged"

print("all conv layer validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 레이어 | forward 출력 shape | backward dx shape | 핵심 특징 |
|---|---|---|---|
| `Conv2d(1,8,K=3,pad=1)` | `(B,8,28,28)` | `(B,1,28,28)` | im2col 행렬 곱, He 초기화 |
| `MaxPool2d(K=2)` | `(B,8,14,14)` | `(B,8,28,28)` | argmax 위치에만 gradient |
| `Flatten` | `(B,8*14*14)` | `(B,8,14,14)` | reshape 왕복 |
| `Dropout(p=0.5)` | `(B,D)` (50% 비활성화) | `(B,D)` | inverted scaling, eval=통과 |

**핵심 설계 원칙**
- `im2col`은 합성곱을 행렬 곱으로 변환하여 numpy/cupy 모두 효율적으로 처리한다.
- `xp` 파라미터로 numpy(CPU)와 cupy(GPU)를 선택하며, `try/except`로 CuPy fallback을 구현한다.
- 모든 레이어는 `Module` 인터페이스를 따르므로 MLP 레이어(`Linear`, `ReLU`)와 `Sequential`로 자유롭게 조합할 수 있다.